# 🚀 Модуль 7. Урок 2. Сплитование трафика, A/A-тесты и Sample Ratio Mismatch (SRM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_02_splitting_aa_tests_srm.ipynb)

## 📌 Бизнес-контекст и методология
Sample Ratio Mismatch (SRM) — это симптом поломки системы сплитования трафика.
Если тест настроен 50/50, но фактически пришло 49 000 и 51 000 пользователей, хи-квадрат тест покажет, случайность это или баг.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np
from scipy import stats

users_a = 49_210
users_b = 50_790
total_users = users_a + users_b
expected = [total_users / 2, total_users / 2]

chi2_stat, srm_p_value = stats.chisquare([users_a, users_b], expected)
print(f"Chi2 stat:   {chi2_stat:.3f}")
print(f"SRM p-value: {srm_p_value:.4f}")

if srm_p_value < 0.01:
    print("🚨 Зафиксирован SRM! Результаты теста невалидны.")
else:
    print("🟢 SRM отсутствует: группы распределены равномерно.")


Chi2 stat:   24.964
SRM p-value: 0.0000
🚨 Зафиксирован SRM! Результаты теста невалидны.


### 🎯 Практическое задание 1: Детекция сильного перекоса
Проверьте выборку 47 000 vs 53 000 на наличие SRM.

In [3]:
### ВАШ КОД ЗДЕСЬ
_, bad_srm_pval = stats.chisquare([47_000, 53_000], [50_000, 50_000])
print(f"p-value для 47k vs 53k: {bad_srm_pval:.4e}")

assert bad_srm_pval < 0.001, "Перекос должен быть статистически значимым"
print("✅ Тест успешно пройден: детектор SRM сработал корректно!")


p-value для 47k vs 53k: 2.8157e-80
✅ Тест успешно пройден: детектор SRM сработал корректно!


### 💼 Причины SRM
1. Редиректы и кэширование на стороне CDN.
2. Боты, попадающие только в одну ветку.
3. Ошибки в хэш-функции соления ID пользователя (salt).